# 01 · Setup do Catalog, Schemas e Volume de Staging

Cria toda a estrutura do Unity Catalog usada pelo projeto **AntecipeAI**:

- 1 catalog dedicado (`antecipeai`, configurável no `.env`)
- 4 schemas **irmãos** dentro dele: `landing`, `bronze`, `silver`, `gold`
- 1 Volume (`raw`) dentro do schema `landing`, usado como staging area
para os arquivos que chegam (hoje: extração única em `.xlsx` da
Locaweb; futuramente: extrações incrementais)

**Decisão de arquitetura:** o schema `landing` guarda o Volume de arquivos
crus (staging), separado da camada `bronze` (que já é Delta). Isso não
estava 100% explícito na conversa anterior — se preferirem outro nome ou
organização para esse schema de staging, é só ajustar `SCHEMA_LANDING` no
`.env` e rodar este notebook de novo (é idempotente, usa `IF NOT EXISTS`
em tudo).

Este notebook é seguro para rodar mais de uma vez (idempotente).

In [0]:
%run ./00_config

# 00 · Configuração do projeto AntecipeAI

Este notebook **não é uma etapa do pipeline** — ele é chamado com `%run` no
início de todos os outros notebooks para carregar a configuração central
do projeto a partir do arquivo `.env`.

A ideia por trás disso: migrar o projeto do Databricks Free (tudo managed,
storage do próprio metastore) para um ambiente de nuvem (S3/AWS,
ADLS/Azure, GCS/GCP, Object Storage/OCI) deve ser uma **troca de valores
no `.env`**, e não uma reescrita de notebook.

## Localizar e carregar o `.env`

Assumimos a estrutura de pastas `antecipeai/notebooks/` e
`antecipeai/config/antecipeai.env` lado a lado no Repo/Workspace. Se a sua
estrutura for diferente, informe o caminho exato no widget
`env_file_path` antes de rodar este notebook.

Configuração carregada de: ../config/antecipeai.env


## Defaults (usados apenas se o `.env` não for encontrado)

## Variáveis expostas para os notebooks que derem `%run` neste

Por ser chamado via `%run`, tudo que é definido aqui fica disponível no
notebook que chamou — não precisa importar nada manualmente depois.

Configuração ativa:
  CATALOG         = antecipeai
  SCHEMA_LANDING  = landing
  SCHEMA_BRONZE   = bronze
  SCHEMA_SILVER   = silver
  SCHEMA_GOLD     = gold
  VOLUME_RAW      = raw
  TABLE_TYPE      = MANAGED
  STORAGE_ROOT    = (vazio - ok p/ MANAGED)
  CLOUD_PROVIDER  = NONE


## Helper: DDL de criação de schema (MANAGED vs EXTERNAL)

Centraliza a única parte do projeto que de fato muda entre "Databricks
Free" e "produção na nuvem": **onde** o schema grava fisicamente os
dados. O resto do código (leituras, transformações, escrita de tabelas)
não muda uma linha.

Helpers disponíveis: qualified_table(schema, table), volume_path(subpath), create_schema_sql(schema_name)


## Criar o catalog

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"Catalog '{CATALOG}' pronto e selecionado como catalog ativo.")

Catalog 'antecipeai' pronto e selecionado como catalog ativo.


## Criar os schemas (camadas) — landing, bronze, silver, gold

O DDL exato muda conforme `ANTECIPEAI_TABLE_TYPE` no `.env`:
- `MANAGED` (default do MVP): sem `LOCATION`, o metastore decide onde gravar.
- `EXTERNAL`: adiciona `MANAGED LOCATION` apontando para o bucket/container
configurado em `ANTECIPEAI_STORAGE_ROOT` — é a troca que vamos fazer se o
projeto for aprovado e migrar para S3.

In [0]:
for schema in [SCHEMA_LANDING, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD]:
    ddl = create_schema_sql(schema)
    print(f"Executando: {ddl}")
    spark.sql(ddl)

print("\nSchemas criados/confirmados:", [SCHEMA_LANDING, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD])

Executando: CREATE SCHEMA IF NOT EXISTS antecipeai.landing
Executando: CREATE SCHEMA IF NOT EXISTS antecipeai.bronze
Executando: CREATE SCHEMA IF NOT EXISTS antecipeai.silver
Executando: CREATE SCHEMA IF NOT EXISTS antecipeai.gold

Schemas criados/confirmados: ['landing', 'bronze', 'silver', 'gold']


## Criar o Volume de staging (`landing.raw`)

É para dentro deste Volume que o arquivo `LW-DATASET.xlsx` (ou, no futuro,
as extrações incrementais em CSV) deve ser enviado manualmente via
**"Upload to this volume"** na UI do Catalog Explorer, antes de rodar o
notebook `02_bootstrap_landing_convert_xlsx`.

In [0]:
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_LANDING}.{VOLUME_RAW}
""")

print(f"Volume pronto em: {volume_path()}")
print("Envie o arquivo LW-DATASET.xlsx para esse volume (Catalog Explorer > Upload) antes do próximo notebook.")

Volume pronto em: /Volumes/antecipeai/landing/raw
Envie o arquivo LW-DATASET.xlsx para esse volume (Catalog Explorer > Upload) antes do próximo notebook.


## Conferência final da estrutura

In [0]:
print("=== Schemas em", CATALOG, "===")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

print("=== Volumes em", f"{CATALOG}.{SCHEMA_LANDING}", "===")
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_LANDING}"))

=== Schemas em antecipeai ===


databaseName
bronze
default
gold
information_schema
landing
silver


=== Volumes em antecipeai.landing ===


database,volume_name
landing,raw
